In [1]:
import pandas as pd
import os
from pathlib import Path
import unicodedata

In [67]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [3]:
def remove_accents(texto: str) -> str:
    nfd = unicodedata.normalize("NFD", texto)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

In [4]:
def get_exercise_type(row):
    tipos = [row.tipo_ejercicio_x, row.tipo_ejercicio_y]
    tipo  = [i for i in tipos if pd.notna(i)]
    if len(tipo) > 0:
        return tipo[0]
    else:
        return pd.NA

## Combinar enunciados con clasificaciones

Cargar enunciados

In [ ]:
def combine(csv_classified, csv_enun = "ejercicios_clasificados.csv"): 
    df_enun = pd.read_csv(root / "csv" / csv_enun, encoding="utf-8")
    
    # set index id
    df_enun = df_enun.set_index("id")

    # load classified exercises to be merged
    df_classified = pd.read_csv(root / "csv" / csv_classified)
    
    # set index id
    df_classified = df_classified.set_index("id")

    # merge classified (df_classified) with already classified exercises (df_enun)
    df_enun.update(df_classified.tipo_ejercicio)

    # restore id to a column
    df_enun = df_enun.reset_index(drop = False)  

    # rearrange columns
    rearranged_cols = ['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
                   'tipo_ejercicio','observaciones', 'enunciado',
                   "id",
                   'info_ejercicio', 'archivo', 'imagen'
                   ]
    df_enun = df_enun[rearranged_cols]

    # save merged df_enun 
    df_enun.to_csv(root / "csv/ejercicios_clasificados.csv",
                   index = False, encoding="utf-8")

    # check if there are exercises with more than
    # one type of exercise
    if any(df_enun.tipo_ejercicio.str.contains("|")):
        df_enun.tipo_ejercicio = df_enun.tipo_ejercicio.str.split("|")

        # explode column tipo_ejercicio
        df_enun = df_enun.explode("tipo_ejercicio")

        # reset index
        df_enun.reset_index(inplace=True, drop = True)
        # look for rows with duplicated id 
        duplicated_indexes = df_enun[df_enun.duplicated(subset = ["id"], keep="first")].index

        # set the new id value for exercises with multiple exercise types
        for idx in duplicated_indexes:
            # get tipo_ejercicio
            tipo_ejercicio = df_enun.loc[idx].tipo_ejercicio

            # clean tipo_ejercicio prior to be inserted in id string
            add_to_id = remove_accents(tipo_ejercicio).replace(" ", "_").lower()

            # get id value
            id = df_enun.loc[idx].id

            # find position of year in id string
            year_pos = id.find("2")
    
            # compose the new id string
            new_id = id[:year_pos] + add_to_id + "_" + id[year_pos:]

            # finally, set the new id value
            df_enun.loc[idx, "id"] = new_id

    # save updated classified exercises
    df_enun.to_csv(root / "csv/ejercicios_clasificados.csv",
                   index = False, encoding="utf-8") 
                          

In [32]:
new_topic = {
"inducción magnética" : [
 'flujo magnético',
 'fuerza entre conductores',
 'experimento de Henry',
 'flujo magnético campo varía tiempo',
 'flujo magnético ángulo varía tiempo',
 'corriente inducida por imán',
 'inducción magnética',
 ],

"campo magnético" : [
 'campo magnético conductor rectilíneo',
 'acelerado diferencia de potencial',
 'campo magnético trayectoria',
 'carga no se desvíe',
 'trayectoria campo magnético',
 ],

 "campo eléctrico" : [
 'distribución de cargas',
 'fuerza campo eléctrico',
 'superficie equipotencial teoría',
 'potencial eléctrico',
 'campo eléctrico',
 'campo eléctrico uniforme'
 ]
}

In [33]:
df_enun.loc[df_enun.asignatura == "Física"].tema.unique()

<ArrowStringArray>
['campo eléctrico y magnético',          'campo gravitatorio',
   'física cuántica y nuclear',                       'ondas',
           'óptica geométrica']
Length: 5, dtype: str

In [38]:
elec_mag_ind_idx= df_enun[(df_enun.asignatura == "Física") & (df_enun.tema == "campo eléctrico y magnético")].index

In [39]:
elec_mag_ind_idx

RangeIndex(start=0, stop=216, step=1)

In [41]:
for idx in elec_mag_ind_idx:
    tipo_ejercicio = df_enun.loc[idx, "tipo_ejercicio"]
    found = False
    for tema, lista_tipos in new_topic.items():
        for tipo in lista_tipos:
            if tipo_ejercicio == tipo:
                df_enun.loc[idx, "tema"] = tema
                found = True
                break
        if found:
            break

In [51]:
df_enun[df_enun.asignatura == "Física"].tema.unique()

<ArrowStringArray>
[      'inducción magnética',           'campo magnético',
           'campo eléctrico',        'campo gravitatorio',
 'física cuántica y nuclear',                     'ondas',
         'óptica geométrica']
Length: 7, dtype: str

## Fix indexes after split

Opcional: insertar enunciados limpios de matrices

In [ ]:
df_clean_st = pd.read_csv(root / "classifier/matrices_limpio.csv")

In [ ]:
df_clean_st.sample()

In [ ]:
df_enun = df_enun.set_index("id")
df_enun.update(df_clean_st.set_index("id").enunciado)

In [ ]:
df_enun[df_enun.tema == "matrices y determinantes"].sample(3)

Exportar enunciados clasificados actualizado

In [ ]:
df_enun.reset_index(drop = False).to_csv(root / "csv/ejercicios_clasificados.csv",
                                         index = False, encoding="utf-8")